# 6.13 Evaluation Metrics

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.13_evaluation-metrics.ipynb)

Companion code for Section 6.13 of the book. Run the cells in order from top to bottom.

**Example 6.13.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, fbeta_score, classification_report,
                              precision_recall_fscore_support)
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Setup
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_tr_s, y_tr)
y_pred = model.predict(X_te_s)
y_prob = model.predict_proba(X_te_s)[:,1]

# ── Compute all metrics ───────────────────────────────────────────
acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, average='weighted')
rec  = recall_score(y_te, y_pred, average='weighted')
f1   = f1_score(y_te, y_pred, average='weighted')
f2   = fbeta_score(y_te, y_pred, beta=2, average='weighted')  # Recall-weighted
f05  = fbeta_score(y_te, y_pred, beta=0.5, average='weighted') # Precision-weighted

print('=== All Evaluation Metrics ===')
print(f'Accuracy:   {acc:.4f}  (overall correct predictions)')
print(f'Precision:  {prec:.4f}  (weighted avg - quality of positive predictions)')
print(f'Recall:     {rec:.4f}  (weighted avg - coverage of actual positives)')
print(f'F1 Score:   {f1:.4f}  (harmonic mean of precision and recall)')
print(f'F2 Score:   {f2:.4f}  (recall-emphasising: β=2)')
print(f'F0.5 Score: {f05:.4f}  (precision-emphasising: β=0.5)')

# ── Per-class breakdown ───────────────────────────────────────────
print('\n=== Per-Class Metrics ===')
prec_cls, rec_cls, f1_cls, sup_cls = precision_recall_fscore_support(y_te, y_pred)
for i, cls in enumerate(data.target_names):
    print(f'{cls:12s}: P={prec_cls[i]:.4f}  R={rec_cls[i]:.4f}  F1={f1_cls[i]:.4f}  n={sup_cls[i]}')

# ── Threshold sensitivity ─────────────────────────────────────────
thresholds = np.linspace(0.1, 0.9, 14)
metrics_by_thresh = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    p = precision_score(y_te, y_pred_t, zero_division=0)
    r = recall_score(y_te, y_pred_t, zero_division=0)
    f = f1_score(y_te, y_pred_t, zero_division=0)
    metrics_by_thresh.append((t, p, r, f))

fig, ax = plt.subplots(figsize=(10, 6))
ts, ps, rs, fs = zip(*metrics_by_thresh)
ax.plot(ts, ps, 'b-o', ms=5, lw=2, label='Precision')
ax.plot(ts, rs, 'r-o', ms=5, lw=2, label='Recall')
ax.plot(ts, fs, 'g-o', ms=5, lw=2, label='F1 Score')
ax.axvline(0.5, color='black', ls='--', alpha=0.7, label='Default threshold (0.5)')
ax.axvline(ts[np.argmax(fs)], color='green', ls=':', lw=2,
          label=f'Best F1 threshold ({ts[np.argmax(fs)]:.2f})')
ax.set_xlabel('Classification Threshold'); ax.set_ylabel('Score')
ax.set_title('Precision, Recall, F1 vs Classification Threshold',
            fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFull Classification Report:')
print(classification_report(y_te, y_pred, target_names=data.target_names))